<a href="https://colab.research.google.com/github/Lucaaa31/Anomaly-Segmentation/blob/master/notebooks/Step8_with_Temperature_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Step 8 Anomaly Segmentation
---


# Settings


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Colab_Projects/Anomaly-Segmentation
#!git pull origin master

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1Pz7ReDC4oIzyB7KLXs9SnUMvmnDqYbyB/Anomaly-Segmentation


## Dependencies

In [ ]:
!pip install --upgrade-strategy only-if-needed -r requirements.txt

## Imports and random seeds

In [ ]:
import os
import glob
import time
import json
import gc
import yaml
import random
import warnings
import importlib
from torch.amp.autocast_mode import autocast
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from lightning import seed_everything
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
from ood_metrics import fpr_at_95_tpr
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F_tf
import torchvision.transforms as T
from torch.utils.data import TensorDataset, DataLoader


# Utils
from utils.build import build_model_and_data
from utils.class_remap import remap_coco_logits_to_cs


seed_everything(42, verbose=False)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
# benchmark must be False together with deterministic=True; for inference the
# speed loss is negligible and runs become reproducible.
torch.backends.cudnn.benchmark = False

---
# Evaluation
## Configuration


In [ ]:
# ===== Configuration =====
# The ONLY knob to change between runs is `model_type`. Run the notebook once per
# model. The table builder uses its own dataset/method/temperature lists, so the
# old per-run dropdowns are gone.
# project_root is taken from the current working dir (the mount cell %cd's into the
# repo), so it works regardless of where the repo lives on Drive.
project_root = os.getcwd()
model_type   = "cityscapes"  # @param ["coco", "cityscapes", "phased", "full"]

_CONFIGS_ROOT = project_root + "/configs/dinov2"
if model_type == "coco":
    _cfg      = _CONFIGS_ROOT + "/coco/panoptic/eomt_base_640_2x.yaml"
    _ckpt     = project_root  + "/models/eomt_coco.bin"
    _override = None
elif model_type == "cityscapes":
    _cfg      = _CONFIGS_ROOT + "/cityscapes/semantic/eomt_base_640.yaml"
    _ckpt     = project_root  + "/models/eomt_cityscapes.bin"
    _override = None
else:
    _cfg      = _CONFIGS_ROOT + "/cityscapes/semantic/eomt_base_640.yaml"
    _ckpt     = project_root  + f"/models/coco_finetune/{model_type}/eomt_finetuned_{model_type}.bin"
    _override = {"img_size": (640, 640)}

# Only used to initialise the model builder (setup_data=False); any valid anomaly
# folder works — the table builder builds its own per-dataset paths.
data_path = project_root + "/dataset/Anomaly_Validation_Dataset/RoadAnomaly21"

# Temperature grid swept to pick the per-dataset "best" T (cityscapes run only).
T_GRID = [0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0, 2.5, 3.0, 3.5]

device = "cuda" if torch.cuda.is_available() else "cpu"

class Args:
    def __init__(self):
        self.model = model_type

args = Args()
print(f"project_root={project_root}")
print(f"model={model_type}  device={device}")
print(f"ckpt:   {_ckpt}")
print(f"config: {_cfg}")

project_root=/content/drive/.shortcut-targets-by-id/1Pz7ReDC4oIzyB7KLXs9SnUMvmnDqYbyB/Anomaly-Segmentation
model=cityscapes  device=cuda
ckpt:   /content/drive/.shortcut-targets-by-id/1Pz7ReDC4oIzyB7KLXs9SnUMvmnDqYbyB/Anomaly-Segmentation/models/eomt_cityscapes.bin
config: /content/drive/.shortcut-targets-by-id/1Pz7ReDC4oIzyB7KLXs9SnUMvmnDqYbyB/Anomaly-Segmentation/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml


We build the model and read its metadata (`num_classes`, `img_size`). Temperature is
applied directly to the logits inside `anomaly_score_from_outputs`, so no wrapper is needed.

In [ ]:
print(f"Building the eomt-{model_type}")
model, meta = build_model_and_data(_cfg, _ckpt, data_path, device, setup_data=False, data_overrides=_override)
print(f"  num_classes={meta.num_classes}  img_size={meta.img_size}")
NUM_CLASSES = meta.num_classes
IMG_SIZE = meta.img_size

Building the eomt-cityscapes


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.


  num_classes=19  img_size=(1024, 1024)


## Utils methods


In [ ]:
def infer_panoptic(img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)

        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits, img_sizes
        )

        preds = model.to_per_pixel_preds_panoptic(
            mask_logits,
            class_logits_per_layer[-1],
            model.stuff_classes,
            model.mask_thresh,
            model.overlap_thresh,
        )[0].cpu()

    pred = preds.numpy()
    sem_pred, inst_pred = pred[..., 0], pred[..., 1]

    target_seg = model.to_per_pixel_targets_panoptic([target])[0].cpu().numpy()
    sem_target, inst_target = target_seg[..., 0], target_seg[..., 1]

    cls_logits = class_logits_per_layer[-1][0][:, :NUM_CLASSES].float()
    masks_probs = torch.sigmoid(mask_logits[0]).float()

    num_q = masks_probs.shape[0]
    H, W = masks_probs.shape[1], masks_probs.shape[2]

    semantic_logits = torch.mm(
        cls_logits.t(),
        masks_probs.view(num_q, -1),
    ).view(NUM_CLASSES, H, W)

    return sem_pred, inst_pred, sem_target, inst_target, semantic_logits, cls_logits, masks_probs

In [ ]:

def draw_black_border(sem, inst, mapping):
    h, w = sem.shape
    out = np.zeros((h, w, 3))
    for s in np.unique(sem):
        out[sem == s] = mapping[s]

    combined = sem.astype(np.int64) * 100000 + inst.astype(np.int64)
    border = np.zeros((h, w), dtype=bool)
    border[1:, :] |= combined[1:, :] != combined[:-1, :]
    border[:-1, :] |= combined[1:, :] != combined[:-1, :]
    border[:, 1:] |= combined[:, 1:] != combined[:, :-1]
    border[:, :-1] |= combined[:, 1:] != combined[:, :-1]
    out[border] = 0
    return out


def plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target):
    all_ids = np.union1d(np.unique(sem_pred), np.unique(sem_target))
    mapping = {
        s: (
            [0, 0, 0]
            if s == -1 or s == model.num_classes
            else plt.cm.hsv(i / len(all_ids))[:3]
        )
        for i, s in enumerate(all_ids)
    }

    vis_pred = draw_black_border(sem_pred, inst_pred, mapping)
    vis_target = draw_black_border(sem_target, inst_target, mapping)

    img_np = (
        img.cpu().numpy().transpose(1, 2, 0) if img.dim() == 3 else img.cpu().numpy()
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title("Input")
    axes[1].imshow(vis_pred)
    axes[1].set_title("Prediction")
    axes[2].imshow(vis_target)
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Post hoc methods

In [ ]:
def get_msp(logits):

    probs = F.softmax(logits.float(), dim=0)
    msp = probs.max(dim=0).values              # [H, W]
    return (1.0 - msp).detach().cpu().numpy().astype("float32")


def get_maxlogit(logits):
    max_logit = logits.float().max(dim=0).values
    return (-max_logit).detach().cpu().numpy().astype("float32")


def get_entropy(logits):

    logits = logits.float()
    probs = F.softmax(logits, dim=0)
    num_classes = logits.shape[0]
    entropy = -(probs * (probs + 1e-7).log()).sum(dim=0)   # [H, W]
    norm_entropy = entropy / torch.log(
        torch.tensor(num_classes, dtype=torch.float32, device=logits.device)
    )
    return norm_entropy.detach().cpu().numpy().astype("float32")


def get_Rba(cls_logits, masks_probs):


    cls_logits  = cls_logits.float()
    masks_probs = masks_probs.float()

    query_class_score = F.softmax(cls_logits, dim=1).max(dim=1).values

    rba_score = (query_class_score[:, None, None] * masks_probs).max(dim=0).values

    return (1.0 - rba_score).detach().cpu().numpy().astype("float32")

## Anomaly score for one image (given a temperature)
This helper applies a temperature `T` to the model outputs and returns the per-pixel anomaly score for the selected post-hoc method. It is used both for normal inference and for the `best`-temperature search, so the scoring logic lives in a single place.

In [ ]:
def anomaly_score_from_outputs(result_logits, cls_logits_raw, masks_probs_raw, T, method):
    """Compute the anomaly score map for one image at temperature T.

    result_logits   : semantic logits  [C, H, W]   (already remapped if COCO)
    cls_logits_raw  : query class logits [Q, C(+1)]
    masks_probs_raw : query mask probs   [Q, H, W]
    T               : scalar temperature (float or 0-dim tensor)
    method          : one of "MSP", "max_logit", "max_entropy", "RbA"
    """
    scaled_logits = result_logits / T
    match method:
        case "MSP":
            return get_msp(scaled_logits)
        case "max_logit":
            return get_maxlogit(scaled_logits)
        case "max_entropy":
            return get_entropy(scaled_logits)
        case "RbA":
            # for RbA temperature is applied to the query class logits
            return get_Rba(cls_logits_raw / T, masks_probs_raw)
    raise ValueError(f"Unknown method: {method}")

## Preprocessing

In [ ]:
def input_transform(img):
    img_resized = F_tf.resize(img, IMG_SIZE, interpolation=T.InterpolationMode.BILINEAR)
    img_tensor = F_tf.to_tensor(img_resized)
    return (img_tensor * 255).to(torch.uint8).to(device)

def target_transform(img):
    img_resized = F_tf.resize(img, IMG_SIZE, interpolation=T.InterpolationMode.NEAREST)
    return np.array(img_resized)

## Anomaly GT remap helper
Same dataset-specific OoD label remapping you already had, factored out so it can be reused by the collection pass and the temperature search.

In [ ]:
# Dataset-specific OoD-label remap and GT-path resolution. Both match on the
# EXACT dataset name (passed in) instead of `in pathGT` — the substring test made
# "RoadAnomaly" also fire for "RoadAnomaly21". Convention after remap:
# 1 = anomaly, 0 = in-distribution, anything else (e.g. 255) = ignore.
# Mask encodings were verified per dataset:
#   RoadAnomaly21 / RoadObsticle21 / fs_static / FS_LostFound_full -> 0/1/255
#   RoadAnomaly -> 0 / 2  (2 = anomaly, remapped to 1)
def remap_ood_gt(target_np, dataset):
    ood_gts = target_np.copy()
    if dataset == "RoadAnomaly":
        ood_gts = np.where((ood_gts == 2), 1, ood_gts)
    # all other datasets already use 0 / 1 / 255 -> no remap
    return ood_gts


def gt_path_for(path, dataset):
    pathGT = path.replace("images", "labels_masks")
    if dataset == "RoadObsticle21":
        pathGT = pathGT.replace("webp", "png")
    elif dataset == "fs_static":
        pathGT = pathGT.replace("jpg", "png")
    elif dataset == "RoadAnomaly":
        pathGT = pathGT.replace("jpg", "png")
    return pathGT

---
# Build the Step-8 tables in one run

For the **selected `model_type`** (Configuration cell), this section streams over each
dataset (one model forward per image) and derives **all four post-hoc methods**
(MSP / MaxLogit / MaxEntropy / RbA) on the **five anomaly benchmarks**. It produces two
separate outputs:

1. **Anomaly detection** (every `model_type`): methods × datasets at baseline **T = 1.0**.
   → `results/step8_anomaly_{model}.txt`
2. **Temperature scaling** (**only `model_type == "cityscapes"`**): **MSP only**, rows
   `T=1.0 / 0.5 / 0.75 / 1.1 / best`, where **best t is a single GLOBAL temperature** —
   the `T_GRID` value with the highest **mean AuPRC over the 5 datasets** (shown in the
   row label, e.g. `MSP (best t=2.5)`).
   → `results/step8_temperature_cityscapes.txt`

Properties:

- **Streaming, not caching** — only the masked per-pixel scores are kept (float16); the heavy
  raw `masks_probs` (~300 MB/image for COCO) are discarded right after use, so RAM stays
  bounded. (Caching the whole dataset is what crashed the session before.)
- **Exact dataset matching** in the GT remap (no `"RoadAnomaly"` firing for `RoadAnomaly21`),
  and FS L&F treated as 0/1/255 like the others.
- For cityscapes, the builder stores **every** evaluated temperature per dataset, so the
  global best t is selected afterwards in the render cell.
- **Per-dataset checkpoint** in `checkpoints/step8/step8_checkpoint_{model}.json` — re-running
  skips finished datasets, surviving Colab disconnects.
- **Time budget** — stops gracefully before the GPU session is killed.

Run once per `model_type`. Temperature scaling runs only on the `cityscapes` run.

**Note:** mIoU is not recomputed (it depends only on the weights). Fill it from Step 4/5.


In [ ]:
def evaluate_dataset(ds_name, method_temps):
    """Stream over one anomaly dataset: ONE model forward per image, then derive
    the requested (method, temperature) scores on the fly.

    `method_temps` is {method: [temperatures]} so we evaluate only what we need:
    the full temperature sweep is requested ONLY for MSP, the other methods only
    at T=1.0. This keeps `score_buf` small (the heavy raw masks_probs are never
    accumulated, and we don't waste RAM sweeping methods we won't report).

    Returns metrics[method][temp] = (auprc%, fpr95%), or None if the dataset has
    no anomaly pixels at all.
    """
    ds_root = project_root + f"/dataset/Anomaly_Validation_Dataset/{ds_name}"
    paths = sorted(glob.glob(os.path.expanduser(ds_root + "/images/*")))

    labels_buf = []
    score_buf = {m: {t: [] for t in ts} for m, ts in method_temps.items()}

    for path in paths:
        images = input_transform(Image.open(path).convert("RGB"))

        gt = gt_path_for(path, ds_name)
        target_np = target_transform(Image.open(gt).convert("L"))

        uids = np.unique(target_np)
        uids = uids[uids != 0]
        if len(uids):
            tdict = {
                "masks": torch.from_numpy(np.stack([target_np == u for u in uids])).bool().to(device),
                "labels": torch.from_numpy(np.asarray(uids)).long().to(device),
            }
        else:
            H0, W0 = target_np.shape
            tdict = {
                "masks": torch.zeros((0, H0, W0), dtype=torch.bool, device=device),
                "labels": torch.zeros((0,), dtype=torch.long, device=device),
            }

        (_, _, _, _, result_logits, cls_logits, masks_probs) = infer_panoptic(images, tdict)
        if args.model == "coco":
            result_logits = remap_coco_logits_to_cs(result_logits)

        ood = remap_ood_gt(target_np, ds_name)
        valid = (ood == 0) | (ood == 1)
        if (ood == 1).sum() == 0:               # no anomaly pixels -> skip image
            del result_logits, cls_logits, masks_probs
            torch.cuda.empty_cache()
            continue

        labels_buf.append(ood[valid].astype("uint8"))
        for m, ts in method_temps.items():
            for t in ts:
                smap = anomaly_score_from_outputs(result_logits, cls_logits, masks_probs, t, m)
                score_buf[m][t].append(smap[valid].astype("float16"))

        del result_logits, cls_logits, masks_probs
        torch.cuda.empty_cache()

    if not labels_buf:
        return None

    labels = np.concatenate(labels_buf)
    metrics = {}
    for m, ts in method_temps.items():
        metrics[m] = {}
        for t in ts:
            scores = np.concatenate(score_buf[m][t]).astype("float32")
            auprc = average_precision_score(labels, scores) * 100.0
            fpr = fpr_at_95_tpr(scores, labels) * 100.0
            metrics[m][t] = (float(auprc), float(fpr))
        score_buf[m] = None  # free as we go
    return metrics

In [ ]:
# ============================================================================
# Single-run Step-8 builder (streaming; no full-dataset cache).
#   * ONE model forward per image; all 4 methods x temps derived on the fly
#   * only masked per-pixel scores are accumulated (float16) -> bounded RAM
#   * ANOMALY DETECTION (baseline T=1.0) is computed for every model_type
#   * TEMPERATURE SCALING is computed ONLY for model_type == "cityscapes":
#       every temperature in EVAL_TEMPS is stored per dataset, so the GLOBAL
#       best T (best mean AuPRC over the 5 datasets) is picked later in cell-27.
#   * per-dataset checkpoint -> resume after a Colab disconnect
#   * time budget -> stop gracefully, finish on the next run
# NOTE: temperature var is `t` here; `T` is the torchvision alias, never reused.
# ============================================================================
METHODS_TABLE  = ["MSP", "max_logit", "max_entropy", "RbA"]
TABLE_DATASETS = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]

COL_LABELS = {
    "RoadAnomaly21":     "SMIYC RA-21",
    "RoadObsticle21":    "SMIYC RO-21",
    "FS_LostFound_full": "FS L&F",
    "fs_static":         "FS Static",
    "RoadAnomaly":       "Road Anomaly",
}

# Temperature scaling only for the Cityscapes model.
DO_TEMPERATURE = (args.model == "cityscapes")

# Temperatures actually evaluated per image. Baseline only when no sweep -> much
# less RAM for coco/phased/full.
if DO_TEMPERATURE:
    EVAL_TEMPS = sorted({float(t) for t in T_GRID} | {1.0, 0.5, 0.75, 1.1})
else:
    EVAL_TEMPS = [1.0]

METHOD_TEMPS = {m: [1.0] for m in METHODS_TABLE}
if DO_TEMPERATURE:
    METHOD_TEMPS["MSP"] = EVAL_TEMPS

CKPT_DIR  = project_root + "/checkpoints/step8"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs("results", exist_ok=True)
CKPT_PATH    = f"{CKPT_DIR}/step8_checkpoint_{args.model}.json"
TIME_LIMIT_S = 4.5 * 3600

def _tkey(t):
    return str(t)

# --- resume from checkpoint -------------------------------------------------
# results[method]["none"][dataset]        = [auprc%, fpr95%, 1.0]   (baseline)
# results[method][str(temp)][dataset]     = [auprc%, fpr95%, temp]  (sweep)
results = {}
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        results = json.load(f)
    print("Resumed checkpoint. Completed datasets:", results.get("_done", []))
results.setdefault("_done", [])

print(f"model={args.model} | temperature scaling: {'ON' if DO_TEMPERATURE else 'OFF (baseline only)'}")

t0 = time.time()
out_of_time = False

for ds in TABLE_DATASETS:
    if ds in results["_done"]:
        print(f"[skip] {ds} already in checkpoint")
        continue
    if time.time() - t0 > TIME_LIMIT_S:
        print(f"[time] budget reached before {ds}; stopping (resume next run)")
        out_of_time = True
        break

    print(f"\n=== {ds} ===")
    metrics = evaluate_dataset(ds, METHOD_TEMPS)

    if metrics is None:
        print(f"[warn] {ds}: no anomaly pixels found in any image; skipping")
        results["_done"].append(ds)
        with open(CKPT_PATH, "w") as fj:
            json.dump(results, fj, indent=2)
        continue

    for method in METHODS_TABLE:
        a, f = metrics[method][1.0]
        results.setdefault(method, {}).setdefault("none", {})[ds] = [a, f, 1.0]
        print(f"  {method:12s} baseline AuPRC={a:6.2f}%  FPR95={f:6.2f}%")

    # sweep di temperature salvato SOLO per MSP (best T globale scelto in cell-27)
    if DO_TEMPERATURE:
        for t in EVAL_TEMPS:
            aa, ff = metrics["MSP"][t]
            results["MSP"].setdefault(_tkey(t), {})[ds] = [aa, ff, t]
        print(f"               stored {len(EVAL_TEMPS)} MSP temperatures")


    results["_done"].append(ds)
    with open(CKPT_PATH, "w") as fj:
        json.dump(results, fj, indent=2)
    print(f"[ckpt] saved -> {CKPT_PATH}")

    del metrics
    gc.collect()
    torch.cuda.empty_cache()

print("\nDone." if not out_of_time else "\nStopped early — re-run this cell to continue.")

Resumed checkpoint. Completed datasets: ['RoadAnomaly21', 'RoadObsticle21']
model=cityscapes | temperature scaling: ON
[skip] RoadAnomaly21 already in checkpoint
[skip] RoadObsticle21 already in checkpoint

=== FS_LostFound_full ===
  MSP          baseline AuPRC= 30.69%  FPR95= 14.94%
  max_logit    baseline AuPRC= 26.44%  FPR95= 16.44%
  max_entropy  baseline AuPRC= 30.24%  FPR95= 15.54%
  RbA          baseline AuPRC= 30.74%  FPR95= 14.69%
               stored 11 MSP temperatures
[ckpt] saved -> /content/drive/.shortcut-targets-by-id/1Pz7ReDC4oIzyB7KLXs9SnUMvmnDqYbyB/Anomaly-Segmentation/checkpoints/step8/step8_checkpoint_cityscapes.json

=== fs_static ===
  MSP          baseline AuPRC= 52.46%  FPR95= 76.86%
  max_logit    baseline AuPRC= 53.40%  FPR95= 77.53%
  max_entropy  baseline AuPRC= 50.00%  FPR95= 76.87%
  RbA          baseline AuPRC= 55.80%  FPR95= 34.39%
               stored 11 MSP temperatures
[ckpt] saved -> /content/drive/.shortcut-targets-by-id/1Pz7ReDC4oIzyB7KLXs9SnUM

In [ ]:
# ============================================================================
# Render the tables from the checkpoint (standalone, no recompute).
#   (1) render_anomaly()         -> 4 methods x 5 datasets, baseline T=1.0
#   (2) render_temperature_msp() -> MSP only: rows MSP / MSP(t=0.5) /
#       MSP(t=0.75) / MSP(t=1.1) / MSP (best t=X). best t = swept temperature
#       with the highest MEAN AuPRC over the 5 datasets (shown in the label).
# ============================================================================
with open(CKPT_PATH) as f:
    results = json.load(f)

MODEL_NAME = f"EoMT-{args.model}"
ROWS = [("MSP", "MSP"), ("MaxLogit", "max_logit"),
        ("MaxEntropy", "max_entropy"), ("RbA", "RbA")]
COLS = [("SMIYC RA-21",  "RoadAnomaly21"),
        ("SMIYC RO-21",  "RoadObsticle21"),
        ("FS L&F",       "FS_LostFound_full"),
        ("FS Static",    "fs_static"),
        ("Road Anomaly", "RoadAnomaly")]
W_MODEL, W_MIOU, W_METHOD, W_CELL = 12, 6, 12, 9


def _cell(method_key, ds_key, temp_key="none"):
    try:
        a, f, _T = results[method_key][temp_key][ds_key]
        return (f"{a:.2f}", f"{f:.2f}")
    except (KeyError, TypeError):
        return ("-", "-")


def render_anomaly():
    """Anomaly-detection table: methods x datasets at baseline T = 1.0."""
    line1 = " " * (W_MODEL + 1 + W_MIOU + 1 + W_METHOD)
    for disp, _ in COLS:
        line1 += " " + disp.center(2 * W_CELL + 1)
    line2 = f"{'Model':<{W_MODEL}} {'mIoU':<{W_MIOU}} {'Method':<{W_METHOD}}"
    for _ in COLS:
        line2 += f" {'AuPRC':>{W_CELL}} {'FPR95':>{W_CELL}}"
    sep = "-" * len(line2)
    body = []
    for i, (rdisp, rkey) in enumerate(ROWS):
        model = MODEL_NAME if i == 0 else ""
        miou  = "----"     if i == 0 else ""
        row = f"{model:<{W_MODEL}} {miou:<{W_MIOU}} {rdisp:<{W_METHOD}}"
        for _, dkey in COLS:
            a, fp = _cell(rkey, dkey, "none")
            row += f" {a:>{W_CELL}} {fp:>{W_CELL}}"
        body.append(row)
    title = f"Step 8 - EoMT anomaly detection - model={MODEL_NAME}, T=1.0"
    return "\n".join([title, "", line1, line2, sep, *body, ""])


def render_temperature_msp():
    """Temperature-scaling table, MSP only. best t over the 5 datasets."""
    msp = results.get("MSP", {})

    best_key, best_mean = None, -1.0
    for k, per_ds in msp.items():
        if k == "none":
            continue
        aucs = [per_ds[dk][0] for _, dk in COLS if dk in per_ds]
        if not aucs:
            continue
        m = sum(aucs) / len(aucs)
        if m > best_mean:
            best_mean, best_key = m, k
    best_t = float(best_key) if best_key is not None else 1.0

    rows = [("MSP",                       "none"),
            ("MSP (t=0.5)",               "0.5"),
            ("MSP (t=0.75)",              "0.75"),
            ("MSP (t=1.1)",               "1.1"),
            (f"MSP (best t={best_t:g})",  best_key)]

    W_M = 20
    line1 = " " * (W_M + 1 + W_MIOU)
    for disp, _ in COLS:
        line1 += " " + disp.center(2 * W_CELL + 1)
    line2 = f"{'Method':<{W_M}} {'mIoU':<{W_MIOU}}"
    for _ in COLS:
        line2 += f" {'AuPRC':>{W_CELL}} {'FPR95':>{W_CELL}}"
    sep = "-" * len(line2)
    body = []
    for rdisp, tkey in rows:
        row = f"{rdisp:<{W_M}} {'----':<{W_MIOU}}"
        for _, dkey in COLS:
            a, fp = _cell("MSP", dkey, tkey) if tkey is not None else ("-", "-")
            row += f" {a:>{W_CELL}} {fp:>{W_CELL}}"
        body.append(row)
    title = f"Step 8 - EoMT temperature scaling (MSP) - model={MODEL_NAME}"
    return "\n".join([title, "", line1, line2, sep, *body, ""])


print(render_anomaly())
if DO_TEMPERATURE:
    print("\n" + render_temperature_msp())
else:
    print(f"\n(no temperature scaling for model={args.model})")


Step 8 - EoMT anomaly detection - model=EoMT-cityscapes, T=1.0

                                     SMIYC RA-21         SMIYC RO-21            FS L&F            FS Static          Road Anomaly   
Model        mIoU   Method           AuPRC     FPR95     AuPRC     FPR95     AuPRC     FPR95     AuPRC     FPR95     AuPRC     FPR95
------------------------------------------------------------------------------------------------------------------------------------
EoMT-cityscapes ----   MSP              71.42     17.64     51.72    100.00     30.69     14.94     52.46     76.86     75.27     32.76
                    MaxLogit         73.11     20.92     48.73    100.00     26.44     16.44     53.40     77.53     70.90     45.01
                    MaxEntropy       71.73     17.60     51.87    100.00     30.24     15.54     50.00     76.87     74.92     32.49
                    RbA              68.64     32.99     79.33      0.82     30.74     14.69     55.80     34.39     75.75     20.27




In [ ]:
# ---- Save result files ------------------------------------------------------
# SAVE_ANOMALY=False -> regenerate ONLY the temperature table, without recreating
# results/step8_anomaly_{model}.txt (use this once the anomaly files have already
# been merged into step8_anomaly_all_results.txt and deleted).
os.makedirs("results", exist_ok=True)

SAVE_ANOMALY = False

if SAVE_ANOMALY:
    ANO_TXT = f"results/step8_anomaly_{args.model}.txt"
    with open(ANO_TXT, "w") as f:
        f.write(render_anomaly())
    print("Saved ->", ANO_TXT)
else:
    print("SAVE_ANOMALY=False -> anomaly file left untouched")

# temperature scaling — MSP only (cityscapes only)
if DO_TEMPERATURE:
    TEMP_TXT = f"results/step8_temperature_{args.model}.txt"
    with open(TEMP_TXT, "w") as f:
        f.write(render_temperature_msp())
    print("Saved ->", TEMP_TXT)
else:
    print(f"(no temperature file for model={args.model})")

SAVE_ANOMALY=False -> anomaly file left untouched
Saved -> results/step8_temperature_cityscapes.txt


In [ ]:
# ============================================================================
# Merge the 4 per-model anomaly tables into ONE combined table, then delete them.
#   in : results/step8_anomaly_{coco,cityscapes,phased,full}.txt
#   out: results/step8_anomaly_all_results.txt
# ============================================================================
import os, re

RES_DIR  = os.path.join(project_root, "results")
OUT_PATH = os.path.join(RES_DIR, "step8_anomaly_all_results.txt")

COLS = [("SMIYC RA-21", "RoadAnomaly21"), ("SMIYC RO-21", "RoadObsticle21"),
        ("FS L&F", "FS_LostFound_full"), ("FS Static", "fs_static"),
        ("Road Anomaly", "RoadAnomaly")]
METHODS     = ["MSP", "MaxLogit", "MaxEntropy", "RbA"]
MODEL_ORDER = ["coco", "cityscapes", "phased", "full"]
MODEL_DISP  = {"coco": "EoMT-COCO", "cityscapes": "EoMT-CS",
               "phased": "EoMT-FT-phased", "full": "EoMT-FT-full"}
NVAL = 2 * len(COLS)

def parse_file(path):
    out = {}
    with open(path) as f:
        for line in f:
            for meth in METHODS:
                if re.search(rf"\b{re.escape(meth)}\b", line):
                    toks = line.split()
                    if len(toks) >= NVAL:
                        out[meth] = toks[-NVAL:]
                    break
    return out

data, found = {}, []
for key in MODEL_ORDER:
    p = os.path.join(RES_DIR, f"step8_anomaly_{key}.txt")
    if os.path.exists(p):
        data[key] = parse_file(p)
        found.append((key, p))

if not data:
    print("No step8_anomaly_{model}.txt files found — nothing to merge.")
else:
    W_MODEL, W_METHOD, W_CELL = 16, 12, 9
    line1 = " " * (W_MODEL + 1 + W_METHOD)
    for disp, _ in COLS:
        line1 += " " + disp.center(2 * W_CELL + 1)
    line2 = f"{'Model':<{W_MODEL}} {'Method':<{W_METHOD}}"
    for _ in COLS:
        line2 += f" {'AuPRC':>{W_CELL}} {'FPR95':>{W_CELL}}"
    sep = "-" * len(line2)

    lines = ["Step 8 — EoMT anomaly detection — all models (baseline T=1.0)",
             "", line1, line2, sep]
    for key in MODEL_ORDER:
        if key not in data:
            continue
        md = data[key]
        for j, meth in enumerate(METHODS):
            model = MODEL_DISP.get(key, key) if j == 0 else ""
            row = f"{model:<{W_MODEL}} {meth:<{W_METHOD}}"
            vals = md.get(meth, ["-"] * NVAL)
            for k in range(len(COLS)):
                row += f" {vals[2*k]:>{W_CELL}} {vals[2*k+1]:>{W_CELL}}"
            lines.append(row)
        lines.append(sep)

    with open(OUT_PATH, "w") as f:
        f.write("\n".join(lines) + "\n")
    print("Merged ->", OUT_PATH, "\n")
    print("\n".join(lines))

    for key, p in found:
        os.remove(p)
        print("Deleted", p)


No step8_anomaly_{model}.txt files found — nothing to merge.
